In [113]:
import json
from pathlib import Path
payload_path = Path("/home/tsuruoka/hdd/BEV/CarlaRunner/DrivingAgent/sample_payload.json")



In [114]:
import numpy as np
import json
import requests


url = "http://localhost:8888/infer"


with payload_path.open() as f:
    payload = json.load(f)

response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()

data = response.json()
print(data["predictions"][0].keys())




dict_keys(['SC_metric', 'SSC_metric', 'pred_c', 'pred_f'])


In [115]:
import torch
import torch.nn.functional as F
import numpy as np


def logits_to_occupancy(pred_c_logits: np.ndarray, target_shape=(40, 512, 512)) -> np.ndarray:
    """Convert OpenOccupancy logits to a labeled grid.

    Mirrors the pipeline in projects/occ_plugin/core/visualizer/show_occ.py:
    1) treat input as (C, H, W, D) or (1, C, H, W, D) logits
    2) reorder to (C, D, H, W) for trilinear upsampling
    3) interpolate to the nuScenes grid (D,H,W) and argmax over classes
    4) return (H, W, D) int16 grid for visualization
    """
    tensor = torch.from_numpy(pred_c_logits)
    if tensor.ndim == 5:
        tensor = tensor[0]
    if tensor.ndim != 4:
        raise ValueError(f"Unexpected pred_c shape: {tuple(tensor.shape)}")

    # reshape (C, H, W, D) -> (C, D, H, W) before interpolation
    tensor = tensor.permute(0, 3, 1, 2).contiguous().unsqueeze(0)
    tensor = F.interpolate(tensor, size=target_shape, mode="trilinear", align_corners=False)
    logits = torch.argmax(tensor[0], dim=0)  # (D, H, W)
    grid = logits.permute(1, 2, 0).contiguous().cpu().numpy().astype(np.int16)  # (H, W, D)
    return grid


raw_pred = np.array(data["predictions"][0]["pred_c"], dtype=np.float32)
print("raw pred shape:", raw_pred.shape)
occ_grid = logits_to_occupancy(raw_pred, target_shape=(40, 512, 512))

unique, counts = np.unique(occ_grid, return_counts=True)
dist = dict(zip(unique.tolist(), counts.tolist()))
print("grid shape:", occ_grid.shape, "dtype:", occ_grid.dtype)
print("class histogram:", dist)

np.save("pred_c.npy", occ_grid)


raw pred shape: (1, 17, 128, 128, 10)
grid shape: (512, 512, 40) dtype: int16
class histogram: {0: 10485760}


In [116]:
import sys
sys.path.append("/home/tsuruoka/hdd/BEV/OpenOccupancy")

from src.utils.occupancy_visualizer import OccupancyGridVisualizer



visualizer = OccupancyGridVisualizer(voxel_size=1.0, z_scale=1.0, default_mode="scatter")
pred_file = "/home/tsuruoka/hdd/BEV/CarlaRunner/DrivingAgent/notebooks/pred_c.npy"
#pred_file = "/home/tsuruoka/hdd/BEV/OpenOccupancy/pred.npy"
visualizer.visualize_file("/home/tsuruoka/hdd/BEV/CarlaRunner/DrivingAgent/notebooks/pred_c.npy", "Pseudo GT - Voxel Points")

grid shape: (512, 512, 40), dtype: int16


ValueError: データが空です